# Piper Vietnamese (sea-g2p) - fine-tune on Colab A100 80GB
Multi-speaker, medium quality (22050 Hz), warmstart from a base checkpoint.

In [ ]:
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Config
REPO_URL   = "https://github.com/<YOUR_USER>/piper1-gpl-seag2p.git"  # TODO: your fork containing this change
DATASET_ZIP = "/content/dagiong.zip"   # upload to /content or change to a Drive path
WORK       = "/content/work"
OUT        = "/content/drive/MyDrive/piper_vi"   # checkpoints/config persisted here
VOICE      = "dagiong"
BASE_REPO  = "rhasspy/piper-checkpoints"  # HF dataset
BASE_CKPT_IN_REPO = "en/en_US/libritts_r/medium/epoch=5461-step=2461000.ckpt"  # TODO: verify with the listing cell
BATCH_SIZE = 64
MAX_EPOCHS = 300
import os; os.makedirs(WORK, exist_ok=True); os.makedirs(OUT, exist_ok=True)

In [ ]:
!apt-get -qq install -y espeak-ng build-essential > /dev/null
!git clone {REPO_URL} {WORK}/piper
%cd {WORK}/piper
!pip install -q -e ".[train,vi]"
!bash build_monotonic_align.sh

In [ ]:
# Sanity check: sea-g2p phonemizer
from piper.phonemize_vietnamese import VietnamesePhonemizer
print(VietnamesePhonemizer().phonemize("Chương hai nghìn tám trăm mười. Bầy Gương."))

In [ ]:
!unzip -q -o {DATASET_ZIP} -d {WORK}/data
!find {WORK}/data -maxdepth 2 | head -20

In [ ]:
# Locate CSV and audio dir, validate
import glob, csv, collections, os
csv_path = sorted(glob.glob(f"{WORK}/data/**/*.csv", recursive=True))[0]
audio_dir = os.path.dirname(csv_path)
rows = list(csv.reader(open(csv_path, encoding="utf-8"), delimiter="|"))
speakers = collections.Counter(r[1] for r in rows)
missing = [r[0] for r in rows if not os.path.exists(os.path.join(audio_dir, r[0]))]
NUM_SPEAKERS = len(speakers)
print(csv_path, audio_dir, len(rows), "rows;", NUM_SPEAKERS, "speakers;", len(missing), "missing")
print(speakers.most_common(10))

In [ ]:
# Download base checkpoint (list files first if the name is wrong)
from huggingface_hub import hf_hub_download, list_repo_files
# print([f for f in list_repo_files(BASE_REPO, repo_type="dataset") if "libritts_r" in f])
BASE = hf_hub_download(BASE_REPO, BASE_CKPT_IN_REPO, repo_type="dataset")
print(BASE)

In [ ]:
# Train. --model.warmstart_ckpt is non-strict, so differing speaker count is fine.
!python3 -m piper.train fit \
  --data.voice_name {VOICE} \
  --data.csv_path {csv_path} \
  --data.audio_dir {audio_dir} \
  --data.phoneme_type vietnamese \
  --data.cache_dir {WORK}/cache \
  --data.config_path {OUT}/{VOICE}.onnx.json \
  --data.num_speakers {NUM_SPEAKERS} \
  --data.batch_size {BATCH_SIZE} \
  --data.num_workers 8 \
  --model.sample_rate 22050 \
  --model.num_speakers {NUM_SPEAKERS} \
  --model.warmstart_ckpt {BASE} \
  --trainer.max_epochs {MAX_EPOCHS} \
  --trainer.default_root_dir {OUT}

In [ ]:
# Resume after a disconnect: add  --ckpt_path <OUT>/lightning_logs/version_N/checkpoints/last.ckpt
# and drop --model.warmstart_ckpt.

In [ ]:
# Export ONNX
import glob
ckpt = sorted(glob.glob(f"{OUT}/lightning_logs/*/checkpoints/last.ckpt"))[-1]
!python3 -m piper.train.export_onnx --checkpoint {ckpt} --output-file {OUT}/{VOICE}.onnx

In [ ]:
# Test synthesis
!echo "Xin chào, đây là giọng đọc thử nghiệm." | python3 -m piper -m {OUT}/{VOICE}.onnx -s 0 -f /content/test.wav
from IPython.display import Audio
Audio("/content/test.wav")